# 04 — Metricas de Distancia: Cosine, Dot Product, Euclidean

A escolha da metrica de distancia afeta diretamente a qualidade da busca semantica.

| Metrica | Formula | Range | Quando Usar |
|---------|---------|-------|-------------|
| **Cosine** | `1 - (A·B)/(|A||B|)` | [0, 2] | Texto — direcao importa, magnitude nao |
| **Dot Product** | `-(A·B)` | (-inf, +inf) | Vetores normalizados — equivale a cosine |
| **Euclidean (L2)** | `sqrt(sum((a-b)^2))` | [0, +inf) | Quando magnitude importa |
| **Manhattan (L1)** | `sum(|a-b|)` | [0, +inf) | Outliers — menos sensivel |

**Regra geral para RAG:** Use **Cosine** com vetores nao-normalizados ou **Dot Product** com vetores normalizados.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

# Frases de exemplo organizadas por grupo semantico
frases = {
    'AI/ML': [
        'Machine learning usa algoritmos para aprender com dados',
        'Deep learning e baseado em redes neurais profundas',
        'Transformers revolucionaram o processamento de linguagem natural',
    ],
    'Culinaria': [
        'Como fazer bolo de chocolate em casa',
        'Receita de feijoada brasileira tradicional',
        'Os melhores restaurantes de sushi em Sao Paulo',
    ],
    'Esportes': [
        'O Brasil ganhou cinco copas do mundo de futebol',
        'LeBron James e um dos maiores basketbolistas da historia',
        'Os Jogos Olimpicos acontecem a cada quatro anos',
    ],
}

todas_frases = [f for grupo in frases.values() for f in grupo]
labels = [grupo for grupo, fs in frases.items() for _ in fs]

# Embeddings nao normalizados e normalizados
embs_raw = model.encode(todas_frases, normalize_embeddings=False)
embs_norm = embs_raw / np.linalg.norm(embs_raw, axis=1, keepdims=True)

print(f'Shape: {embs_raw.shape}')
print(f'Norma media (raw):  {np.linalg.norm(embs_raw, axis=1).mean():.4f}')
print(f'Norma media (norm): {np.linalg.norm(embs_norm, axis=1).mean():.4f}')

## 4.1 Cosine Similarity

Mede o **angulo** entre dois vetores. Ignora a magnitude.

```
cosine_sim(A, B) = (A · B) / (|A| * |B|)
```

- `cosine_sim = 1.0` → vetores identicos (angulo = 0°)
- `cosine_sim = 0.0` → vetores ortogonais (angulo = 90°)
- `cosine_sim = -1.0` → vetores opostos (angulo = 180°)

In [ ]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def dot_product_similarity(a, b):
    return np.dot(a, b)

def euclidean_distance(a, b):
    return np.linalg.norm(a - b)

def manhattan_distance(a, b):
    return np.sum(np.abs(a - b))

# Comparar metricas para um par de frases
exemplos = [
    (0, 1, 'ML vs Deep Learning (similar)'),
    (0, 3, 'ML vs Culinaria (diferente)'),
    (3, 4, 'Bolo vs Feijoada (similar de categoria)'),
    (0, 6, 'ML vs Futebol (muito diferente)'),
]

print(f'{'Par':<40} {'Cosine':>8} {'Dot(raw)':>10} {'Euclidean':>11} {'Manhattan':>11}')
print('-' * 82)

for i, j, desc in exemplos:
    cos = cosine_similarity(embs_raw[i], embs_raw[j])
    dot = dot_product_similarity(embs_raw[i], embs_raw[j])
    euc = euclidean_distance(embs_raw[i], embs_raw[j])
    man = manhattan_distance(embs_raw[i], embs_raw[j])
    print(f'{desc:<40} {cos:>8.3f} {dot:>10.3f} {euc:>11.3f} {man:>11.3f}')

## 4.2 Dot Product vs Cosine: Qual a diferenca?

Com vetores **normalizados** (norma = 1), Dot Product == Cosine Similarity.

Com vetores **nao normalizados**, eles diferem:
- **Cosine**: ignora magnitude, so direcao
- **Dot Product**: combina direcao + magnitude

In [ ]:
# Demonstrar com vetores 2D (facil de visualizar)
A = np.array([2.0, 1.0])   # vetor curto
B = np.array([6.0, 3.0])   # vetor 3x mais longo, MESMA DIRECAO
C = np.array([1.0, 2.0])   # vetor diferente

print('Vetores 2D (para visualizacao):')
print(f'A = {A} (norma = {np.linalg.norm(A):.2f})')
print(f'B = {B} (norma = {np.linalg.norm(B):.2f}) <- 3x maior que A, MESMA DIRECAO')
print(f'C = {C} (norma = {np.linalg.norm(C):.2f})')

print(f'\n--- Cosine Similarity ---')
print(f'cos(A, B) = {cosine_similarity(A, B):.4f}  (1.0 = direcao identica)')
print(f'cos(A, C) = {cosine_similarity(A, C):.4f}  (menor = direcao diferente)')

print(f'\n--- Dot Product ---')
print(f'dot(A, B) = {dot_product_similarity(A, B):.4f}  (maior porque B tem magnitude maior!)')
print(f'dot(A, C) = {dot_product_similarity(A, C):.4f}')

print(f'\nConclusao:')
print(f'  Para TEXTO com normalizacao: Cosine e Dot Product dao o mesmo ranking')
print(f'  Para textos com importancias diferentes (ex: popularidade): Dot Product considera isso')

In [ ]:
# Visualizacao 2D das metricas
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Plot 1: Vetores 2D
ax = axes[0]
for vec, name, color in [(A, 'A', 'blue'), (B, 'B', 'red'), (C, 'C', 'green')]:
    ax.quiver(0, 0, vec[0], vec[1], angles='xy', scale_units='xy', scale=1,
              color=color, label=f'{name}={vec}', linewidth=2)
    ax.text(vec[0] + 0.1, vec[1] + 0.1, name, color=color, fontsize=12, fontweight='bold')

# Arco para angulos
theta = np.linspace(0, np.arctan2(A[1], A[0]), 50)
r = 0.5
ax.plot(r * np.cos(theta), r * np.sin(theta), 'b-', alpha=0.5)

ax.set_xlim(-0.5, 7)
ax.set_ylim(-0.5, 4)
ax.set_title('Vetores 2D: Cosine ignora magnitude\nA e B tem MESMA DIRECAO (cos=1.0)', fontsize=11)
ax.set_xlabel('Dimensao 1')
ax.set_ylabel('Dimensao 2')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_aspect('equal')

# Plot 2: Heat map de similaridade
ax = axes[1]
sim_matrix = embs_norm @ embs_norm.T  # cosine com vetores normalizados

cores_grupos = ['#e74c3c'] * 3 + ['#3498db'] * 3 + ['#2ecc71'] * 3
im = ax.imshow(sim_matrix, cmap='RdYlGn', vmin=-0.3, vmax=1.0)
plt.colorbar(im, ax=ax, label='Cosine Similarity')

ax.set_xticks(range(9))
ax.set_yticks(range(9))
ax.set_xticklabels([f'{l[:15]}' for l in todas_frases], rotation=45, ha='right', fontsize=7)
ax.set_yticklabels([f'{l[:15]}' for l in todas_frases], fontsize=7)
ax.set_title('Heatmap de Cosine Similarity\n(verde = similar, vermelho = diferente)', fontsize=11)

# Linhas separando grupos
for pos in [2.5, 5.5]:
    ax.axhline(y=pos, color='black', linewidth=2)
    ax.axvline(x=pos, color='black', linewidth=2)

plt.tight_layout()
plt.show()

## 4.3 Euclidean Distance: Quando magnitude importa

In [ ]:
# Demonstrar quando euclidean pode dar resultados diferentes de cosine
np.random.seed(42)

# Criar embeddings artificiais com magnitudes muito diferentes
frases_test = [
    'Inteligencia artificial',       # 0 - topic: AI
    'Machine learning e IA',          # 1 - topic: AI (muito similar a 0)
    'Algoritmos de aprendizagem',    # 2 - topic: AI
]

embs_test_norm = model.encode(frases_test, normalize_embeddings=True)

# Modificar artificialmente a magnitude de um vetor
embs_test_raw = embs_test_norm.copy()
embs_test_raw[2] = embs_test_raw[2] * 3.0  # escalar o terceiro 3x

query_idx = 0

print('Comparando metricas com magnitude artificial diferente:')
print(f'Query: "{frases_test[query_idx]}"')
print(f'\nVetor 0: norma = {np.linalg.norm(embs_test_raw[0]):.3f}')
print(f'Vetor 1: norma = {np.linalg.norm(embs_test_raw[1]):.3f}  (similar ao 0)')
print(f'Vetor 2: norma = {np.linalg.norm(embs_test_raw[2]):.3f}  (MAGNITUDE AUMENTADA 3x)')

for j in [1, 2]:
    cos = cosine_similarity(embs_test_raw[0], embs_test_raw[j])
    dot = dot_product_similarity(embs_test_raw[0], embs_test_raw[j])
    euc = euclidean_distance(embs_test_raw[0], embs_test_raw[j])
    print(f'\n  vs "{frases_test[j]}":')
    print(f'    Cosine:    {cos:.4f} (apenas direcao)')
    print(f'    Dot:       {dot:.4f} (direcao + magnitude)')
    print(f'    Euclidean: {euc:.4f} (distancia absoluta)')

print('\nNota: para embeddings de texto, SEMPRE normalize para cosine/dot equivalentes')

## 4.4 Configurando Metricas no Qdrant

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams

client = QdrantClient(host='localhost', port=6333)

metricas = {
    'cosine_collection': Distance.COSINE,
    'dot_collection': Distance.DOT,
    'euclid_collection': Distance.EUCLID,
}

for nome, distancia in metricas.items():
    client.recreate_collection(
        collection_name=nome,
        vectors_config=VectorParams(size=384, distance=distancia)
    )
    print(f'Collection {nome} criada com metrica {distancia}')

print('\nNotas importantes sobre Qdrant:')
print('  COSINE: calcula 1 - cosine_sim (menor = mais similar)')
print('  DOT: negativo do produto escalar (menor = mais similar)')
print('  EUCLID: distancia L2 (menor = mais similar)')

## Resumo: Qual metrica escolher?

```
Tipo de dado?
├── Texto semantico (maioria dos casos RAG)
│   └── COSINE (ou DOT com vetores normalizados)
│       Por que? Direcao = significado; magnitude = artefato do modelo
│
├── Vetores com significado de magnitude (ex: popularidade * embedding)
│   └── DOT PRODUCT
│       Por que? Combina relevancia semantica com importancia
│
├── Features numericas (preco, tamanho, coordenadas)
│   └── EUCLIDEAN
│       Por que? Magnitude das features tem significado real
│
└── Features esparsas, muitos outliers
    └── MANHATTAN (L1)
        Por que? Menos sensivel a outliers que L2
```

**Para RAG:** Use `Distance.COSINE` no Qdrant com vetores normalizados.

## Proximo
- [05 — Models Comparison](05_models_comparison.ipynb)